In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
from pyspark.sql.functions import pandas_udf, col
from sklearn.metrics import classification_report, f1_score

In [2]:
spark = SparkSession.builder.appName("nlp-pipeline").getOrCreate()

In [3]:
df = spark.read.csv("../data/raw/Books_rating.csv", header=True, inferSchema=True)

In [4]:
df.printSchema()
df.show(5)
df.count()

root
 |-- Id: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- User_id: string (nullable = true)
 |-- profileName: string (nullable = true)
 |-- review/helpfulness: string (nullable = true)
 |-- review/score: string (nullable = true)
 |-- review/time: string (nullable = true)
 |-- review/summary: string (nullable = true)
 |-- review/text: string (nullable = true)

+----------+--------------------+-----+--------------+--------------------+------------------+------------+-----------+--------------------+--------------------+
|        Id|               Title|Price|       User_id|         profileName|review/helpfulness|review/score|review/time|      review/summary|         review/text|
+----------+--------------------+-----+--------------+--------------------+------------------+------------+-----------+--------------------+--------------------+
|1882931173|Its Only Art If I...| NULL| AVCGYZL8FQQTD|"Jim of Oz ""jim-...|               7/

3000000

In [5]:
df = df.withColumn("Price", col("Price").cast("float")) \
       .withColumn("review/score", col("review/score").cast("float"))

In [6]:
df = df.withColumn("review/time", col("review/time").cast("long"))

In [7]:
df.printSchema()

root
 |-- Id: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Price: float (nullable = true)
 |-- User_id: string (nullable = true)
 |-- profileName: string (nullable = true)
 |-- review/helpfulness: string (nullable = true)
 |-- review/score: float (nullable = true)
 |-- review/time: long (nullable = true)
 |-- review/summary: string (nullable = true)
 |-- review/text: string (nullable = true)



In [8]:
df = df.filter(col("review/score").isNotNull())

In [9]:
df.filter(col("review/score").isNull()).count()

0

In [10]:
df = df.filter((col("review/score") >= 0) & (col("review/score") <= 5))

In [11]:
df.groupBy("review/score").count().orderBy("review/score").show()

+------------+-------+
|review/score|  count|
+------------+-------+
|         1.0| 201000|
|         2.0| 150449|
|         3.0| 252940|
|         4.0| 581728|
|         5.0|1795795|
+------------+-------+



In [12]:
df.filter(col("review/text").isNull()).count()

9

In [13]:
df = df.filter(col("review/text").isNotNull())

In [14]:
df.filter(col("review/text").isNull()).count()

0

In [15]:
df = df.withColumn("review/text", lower("review/text"))

In [16]:
df = df.withColumn(
    "review/text",
    regexp_replace("review/text", r"&[a-zA-Z]+;", "")
)

In [17]:
df.select("review/text").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [18]:
tokenizer = AutoTokenizer.from_pretrained("../model/base_model")

In [19]:
model = AutoModelForSequenceClassification.from_pretrained("../model/base_model")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [20]:
print(model.config.id2label)

{0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2'}


In [21]:
# --- model on GPU, half precision, eval mode ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
if device == "cuda":
    model.half()
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [22]:
id2label = {0: "negative", 1: "neutral", 2: "positive"}


In [23]:
def predict_batch(texts, batch_size=32):
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            chunk = texts[i:i + batch_size]
            inputs = tokenizer(
                chunk,
                padding=True,
                truncation=True,
                max_length=64,        
                return_tensors="pt",
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits
            all_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
    return all_preds

In [24]:
@pandas_udf("integer")
def predict_label(texts: pd.Series) -> pd.Series:
    return pd.Series(predict_batch(texts.tolist()))

In [28]:
test_df = (
    df.dropna(subset=["review/score", "review/text"])
      .limit(300)
      .select("review/score", "review/text")
      .cache()
)
test_df.count()

300

In [29]:
results = (
    test_df
    .withColumn("pred", predict_label(col("review/text")))
    .select("review/score", "pred")
    .toPandas()
)

In [30]:
# --- true labels: same mapping as training ---
score_map = {1: 0, 2: 0, 3: 1, 4: 2, 5: 2}
results["true"] = results["review/score"].astype(float).astype(int).map(score_map)

print(classification_report(
    results["true"], results["pred"],
    target_names=list(id2label.values()),
))
print("Macro-F1:", f1_score(results["true"], results["pred"], average="macro"))

              precision    recall  f1-score   support

    negative       0.72      0.70      0.71        30
     neutral       0.39      0.57      0.46        21
    positive       0.96      0.92      0.94       249

    accuracy                           0.88       300
   macro avg       0.69      0.73      0.70       300
weighted avg       0.89      0.88      0.88       300

Macro-F1: 0.7046993882805467
